## 5. Data Modelling
The models have been selected and saved, it will be run on the dataset and extract topic scores for each year, region and constituency and save this to csv files. 

In [56]:
# loading data
import pandas as pd
import pyarrow.parquet as pq

#for loading model and making models
from gensim import corpora
from gensim.models import LdaModel

#validation
import numpy as np


In [25]:
# import in the data from parquet files and csv
petitions_metadata_df = pd.read_csv('data\petitions_metadata.csv')

petitions_table = pq.read_table('data\petitions_processed_data.parquet')
countries_table = pq.read_table('data\petitions_country_data.parquet')
regions_table = pq.read_table('data\petitions_region_data.parquet')
constituencies_table = pq.read_table('data\petitions_constituency_data.parquet')

petitions_df = petitions_table.to_pandas()
countries_df = countries_table.to_pandas()
regions_df = regions_table.to_pandas()
constituencies_df = constituencies_table.to_pandas()

sample_df = petitions_df[petitions_df['created_at'].dt.year==2021]
processed_sample = sample_df['clean_text'].values

## Load model and test small sample

In [26]:
def load_model(year):
    """ 
    Function to load the relevant model for a given year
    :param year: integer between 2015-2025
    :output: dictionary, corpus and model for the inputted year
    """
    dictionary = corpora.Dictionary.load(f"models/model_{year}/dictionary.dict")
    corpus = corpora.MmCorpus(f"models/model_{year}/corpus.mm")
    model = LdaModel.load(f"models/model_{year}/model.gensim")

    return dictionary, corpus, model

def load_file(year):
    """ 
    Function to load the relevant filtered petition file for a given year
    :param year: integer between 2015-2025
    :output: file for the inputted year
    """
    file = petitions_df[petitions_df['created_at'].dt.year == year]

    return file

In [27]:
# giving topics a generic label for the csv file
topic_labels = {
    0: "Topic 1",
    1: "Topic 2",
    2: "Topic 3",
    3: "Topic 4",
    4: "Topic 5", 
    5: "Topic 6", 
    6: "Topic 7",
    7: "Topic 8",
    8: "Topic 9",
    9: "Topic 10",
    10: "Topic 11",
    11: "Topic 12",
    12: "Topic 13",
    13: "Topic 14",
    14: "Topic 15",
    15: "Topic 16",
    16: "Topic 17",
    17: "Topic 18"
}

In [ ]:
# will be examining how well the model performs for item 150
processed_sample[150]

array(['prioritise', 'receiving', 'vaccination', 'workplace', 'closed',
       'setting', 'remained', 'enabling', 'youngest', 'receiving',
       'socially', 'distance', 'nature', 'requires', 'contact',
       'intimate', 'wearing', 'ppe', 'recommended', 'afforded',
       'vaccination'], dtype=object)

In [29]:
# Check which topic our test document belongs to using the LDA Bag of Words model. Set document num: 
document_num = 150

dictionary, corpus, model = load_model(2021)

# looping through document to get the topics with highest probability
for index, score in sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, model.print_topic(index, 5)))


Score: 0.8022282719612122	 
Topic: 0.032*"quarantine" + 0.025*"vaccinated" + 0.023*"red" + 0.019*"vaccine" + 0.018*"mask"

Score: 0.15872056782245636	 
Topic: 0.017*"nursery" + 0.017*"cancer" + 0.016*"setting" + 0.013*"special" + 0.011*"visit"


Topic assigned is predominantly regarding quarantine and vaccine. Secondary is one regarding nurseries. The former definitely makes sense and there is some intuitive understanding to the second given the keywords selected.
## Testing topic assignment
Checking thresholds for topic assignment, length of file vs corpus and coverage of the top 3 topics.

In [30]:
def validate_corpus_file_alignment(file, corpus, sample_size=10):
    """
    Verify that corpus indices match file indices
    """
    if len(corpus) != len(file):
        print(f"WARNING: Corpus length ({len(corpus)}) != File length ({len(file)})")
        return False
    
    # Sample random documents and verify they make sense together
    import random
    random.seed(42)
    sample_indices = random.sample(range(min(len(corpus), len(file))), min(sample_size, len(corpus)))
    
    for idx in sample_indices:
        # Get original text
        original_text = file.iloc[idx]['clean_text']
        # Get corpus representation
        doc_bow = corpus[idx]
        
        print(f"\nDocument {idx}:")
        print(f"Original text (first 100 chars): {' '.join(original_text[:100])}")
        print(f"BOW length: {len(doc_bow)}")
        
    return True


In [31]:
def validate_topic_assignment(model, corpus, document_num, threshold=0.3):
    """
    Check if a document has a clear dominant topic
    :param threshold: minimum probability for dominant topic
    :return: dict with assignment quality metrics
    """
    topic_dist = model[corpus[document_num]]
    sorted_topics = sorted(topic_dist, key=lambda x: -x[1])
    
    dominant_prob = sorted_topics[0][1]
    second_prob = sorted_topics[1][1] if len(sorted_topics) > 1 else 0
    
    validation = {
        'document_num': document_num,
        'dominant_topic': sorted_topics[0][0],
        'dominant_prob': dominant_prob,
        'probability_gap': dominant_prob - second_prob,
        'is_clear_assignment': dominant_prob > threshold and (dominant_prob - second_prob) > 0.15,
        'entropy': -sum(p * np.log(p) if p > 0 else 0 for _, p in topic_dist)
    }
    
    return validation

In [32]:
possible_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# looping through years to get results
for year in possible_years:
    dictionary, corpus, model = load_model(year)
    file = load_file(year)

    print(f'{year}: corpus file alignment is {validate_corpus_file_alignment(file, corpus)}')

    unclear_assignments = []
    for doc_num in range(min(100, len(corpus))):  # Test first 100 docs
        val = validate_topic_assignment(model, corpus, doc_num)
        if not val['is_clear_assignment']:
            unclear_assignments.append(val)

    print(f"Documents with unclear assignments: {len(unclear_assignments)}")

2015: corpus file alignment is False
Documents with unclear assignments: 19

Document 912:
Original text (first 100 chars): release eu migrant insurance revealed minister hmrc refusing release active insurance issued eu migrant deem unhelpful eu renegotiation unacceptable released eu jonathan porte director institute requested electorate eu denied spirit fairness hiding electorate detrimental preferred
BOW length: 14

Document 204:
Original text (first 100 chars): litter clean overdue cheap simple clearing alarming litter detail promote creation nationwide voluntary litter clean publicly funded planning organising litter pick pick utilise volunteer organised controlled cleaned
BOW length: 16

Document 2253:
Original text (first 100 chars): exit eu june voting eu chose eu departure electorate exit eu regime negotiate eu interact eu voter knowing voting mandate negotiate eu exit final
BOW length: 11

Document 2006:
Original text (first 100 chars): disabled parking location ownership park

In [ ]:
def test_threshold_sensitivity(file, model, corpus, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """
    Test how threshold affects topic assignment counts
    :param file: petition count file to run tests on
    :param model: model for the assigned file
    :param corpus: corpus for the assigned file
    :param thresholds: list of decimals between 0 and 1 which will be used as possible thresholds
    :return: data frame of results including number of assigned and unassigned docs and topic distribution
    """
    results = {}
    
    for threshold in thresholds:
        topic_counts = [0] * 18
        assigned_docs = 0
        
        for document_num in range(len(file)):
            if document_num < len(corpus):
                topic_probs = sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1])
                
                # Check if dominant topic exceeds threshold
                if topic_probs[0][1] > threshold:
                    topic_counts[topic_probs[0][0]] += 1
                    assigned_docs += 1
        
        results[threshold] = {
            'assigned_docs': assigned_docs,
            'unassigned_docs': len(file) - assigned_docs,
            'assignment_rate': assigned_docs / len(file),
            'topic_distribution': topic_counts
        }
    
    return pd.DataFrame(results).T

# Test threshold sensitivity on sample
dictionary, corpus, model = load_model(2023)
file = load_file(2023)

threshold_analysis = test_threshold_sensitivity(file, model, corpus)
print(threshold_analysis)

    assigned_docs unassigned_docs assignment_rate  \
0.3          3002              59        0.980725   
0.4          2709             352        0.885005   
0.5          2162             899        0.706305   
0.6          1565            1496        0.511271   
0.7          1104            1957        0.360666   

                                    topic_distribution  
0.3  [173, 158, 98, 131, 183, 157, 111, 172, 174, 2...  
0.4  [158, 144, 86, 118, 169, 142, 100, 157, 153, 1...  
0.5  [131, 118, 68, 94, 138, 121, 74, 124, 116, 150...  
0.6  [95, 71, 51, 75, 108, 83, 50, 87, 82, 105, 75,...  
0.7  [66, 54, 38, 61, 83, 50, 34, 56, 68, 69, 57, 7...  


In [ ]:
# below code takes a petitions for a given topic and saves to a dataframe to then be examined further to understand topic groupings

def split_topic_petitions(file, topic_num, model, corpus):
    """ 
    Function to split petitions into given topics and save the results to a csv file
    :param file: filtered petition file for a given year
    :param topic_num: integer for the topic
    :param model: model for the given year
    :param corpus: corpus for the given year
    :output: file of petitions with the assigned topic
    """

    # setting up empty dataframe for petitions
    topic_df = pd.DataFrame(columns=['petition_id', 'document', 'topic', 'topic prob', 'signature_count'])

    # loop through all documents
    for document_num in range(len(file)):
        # looping through topic probabilities to get the topics with highest probability
        for index, score in model[corpus[document_num]]:
            # only want documents categorised as given topic
            if (index + 1 == topic_num) & (score>0.5):
                # print("\nScore: {}\t \nTopic: {}".format(score, model.print_topic(index, 5)))
                # saving topics in dictionary 
                file_dict = {
                    'petition_id': file.iloc[document_num, 0],
                    'document': file.iloc[document_num, 6],
                    'topic': index + 1,
                    'topic prob': score,
                    'signature_count': file.iloc[document_num, 4]
                }
                #appending to end of dataframe
                topic_df.loc[len(topic_df)] = file_dict

    return topic_df

In [ ]:
dictionary, corpus, model = load_model(2021)
petitions = load_file(2021)
extracted_df = split_topic_petitions(petitions, 9, model, corpus)
extracted_df.to_csv("data/2021_09.csv")

So far, the breakdown of topics is ok but this doesn't tell us which topics have more significance or importance. This is just a division of all written petitions not taking into account so far that some petitions get a lot more signatures than others and so should be deemed more significant. For this, we need to weight categories based on signatures. To measure the impact of topics, each petition is counted and given a weight of one, to then measure the impact of each petition, this weight is multiplied by the number of signatures. A petition with 1000 signatures will add a score of 1000 to that topic.

In [ ]:
def find_topic_prob(file, model, corpus):
    """
    This function loops through each document for a given year and weights each petition by its signature count.
    :param file: takes the pandas dataframe for a given year with documents listed
    :param model: model for given year
    :param corpus: corpus for given year
    :return: returns a dataframe with added column of topic prob and a list of any errors
    """
    errors = []
    processed_docs = 0
    topic_df = pd.DataFrame(columns=['petition_id', 'document', 'topic', 'topic prob', 'signature_count'])

    # loop through each petition
    for document_num in range(len(file)):
        try:
            # Check corpus bounds
            if document_num >= len(corpus):
                errors.append(f"Doc {document_num}: Out of corpus bounds")
                continue
        
            
            # Get topic probabilities
            topic_probs = model[corpus[document_num]]
            if not topic_probs:
                errors.append(f"Doc {document_num}: Empty topic distribution")
                continue
            
            for topic_id, prob in topic_probs:
                file_dict = {
                'petition_id': file.iloc[document_num, 0],
                'document': file.iloc[document_num, 6],
                'topic': topic_id + 1,
                'topic prob': prob,
                'signature_count': file.iloc[document_num, 4]
                }
                topic_df.loc[len(topic_df)] = file_dict
            processed_docs += 1
            
        except Exception as e:
            errors.append(f"Doc {document_num}: {str(e)}")

    # Log results
    print(f"\nProcessing Summary:")
    print(f"Total documents: {len(file)}")
    print(f"Successfully processed: {processed_docs}")
    print(f"Errors: {len(errors)}")

    if errors:
        print("\nFirst 5 errors:")
        for error in errors[:5]:
            print(f"  - {error}")
    
    return topic_df, errors

In [ ]:
def calculate_weighting_overall(file, year):
    """
    This function loops through each document for a given year and if the topic assigned has probability higher than 0.5, multiplies by signature count 
    and adds to relevant topic weighting.
    :param file: takes the pandas dataframe for a given year with documents listed
    :param model: model for given year
    :param corpus: corpus for given year
    :return: returns a dataframe with topic counts and the list of topic weightings
    """
    top_topics = {}
    topic_weighting = [0] * 18

    # loop through each petition
    for index, row in file.iterrows():
        # find probability
        prob = row['topic prob']

        if prob>=0.5:
            # assign relevant topic num and add weighting to relevant list item
            topic_id = row['topic'] - 1
            prob = row['topic prob']
            num_signatures = row['signature_count']
            topic_weighting[topic_id] += num_signatures * prob

    # when run through all petitions update the dataframe for that year
    top_topics.update({
            "Year": year,
            "Topic 1 Count": topic_weighting[0],
            "Topic 2 Count": topic_weighting[1],
            "Topic 3 Count": topic_weighting[2],
            "Topic 4 Count": topic_weighting[3],
            "Topic 5 Count": topic_weighting[4],
            "Topic 6 Count": topic_weighting[5],
            "Topic 7 Count": topic_weighting[6],
            "Topic 8 Count": topic_weighting[7],
            "Topic 9 Count": topic_weighting[8],
            "Topic 10 Count": topic_weighting[9],
            "Topic 11 Count": topic_weighting[10],
            "Topic 12 Count": topic_weighting[11],
            "Topic 13 Count": topic_weighting[12],
            "Topic 14 Count": topic_weighting[13],
            "Topic 15 Count": topic_weighting[14],
            "Topic 16 Count": topic_weighting[15],
            "Topic 17 Count": topic_weighting[16],
            "Topic 18 Count": topic_weighting[17],
            })
            
    return top_topics, topic_weighting

In [ ]:
def calculate_weighting_con(file, year, con):
    """
    This function loops through each document for a given year and constituency and if the topic assigned has probability higher than 0.5, multiplies by signature count 
    and adds to relevant topic weighting.
    :param file: takes the pandas dataframe for a given year and constituency with documents listed
    :param model: model for given year
    :param corpus: corpus for given year
    :return: returns a dataframe with topic counts and the list of topic weightings
    """
    top_topics = {}
    topic_weighting = [0] * 18

    # filter file by relevant constituency
    filtered_file = file[file['name']==con]

    # loop through petitions for that constituency
    for index, row in filtered_file.iterrows():
        # find probability petiiton belongs to given topic
        prob = row['topic prob']
        if prob>=0.5:
            # assign relevant topic num and add weighting to relevant list item
            topic_id = row['topic'] - 1
            prob = row['topic prob']
            num_signatures = row['area_signature_count']
            topic_weighting[topic_id] += num_signatures * prob

    # when run through all petitions update the dataframe for that year
    top_topics.update({
            "Constituency": con,
            "Year": year,
            "Topic 1 Count": topic_weighting[0],
            "Topic 2 Count": topic_weighting[1],
            "Topic 3 Count": topic_weighting[2],
            "Topic 4 Count": topic_weighting[3],
            "Topic 5 Count": topic_weighting[4],
            "Topic 6 Count": topic_weighting[5],
            "Topic 7 Count": topic_weighting[6],
            "Topic 8 Count": topic_weighting[7],
            "Topic 9 Count": topic_weighting[8],
            "Topic 10 Count": topic_weighting[9],
            "Topic 11 Count": topic_weighting[10],
            "Topic 12 Count": topic_weighting[11],
            "Topic 13 Count": topic_weighting[12],
            "Topic 14 Count": topic_weighting[13],
            "Topic 15 Count": topic_weighting[14],
            "Topic 16 Count": topic_weighting[15],
            "Topic 17 Count": topic_weighting[16],
            "Topic 18 Count": topic_weighting[17],
            })
            
    return top_topics

In [ ]:
def calculate_weighting_region(file, year, region):
    """
    This function loops through each document for a given year and region and if the topic assigned has probability higher than 0.5, multiplies by signature count 
    and adds to relevant topic weighting.
    :param file: takes the pandas dataframe for a given year and region with documents listed
    :param model: model for given year
    :param corpus: corpus for given year
    :return: returns a dataframe with topic counts and the list of topic weightings
    """
    top_topics = {}
    topic_weighting = [0] * 18

    filtered_file = file[file['name']==region]
    for index, row in filtered_file.iterrows():
        prob = row['topic prob']
        if prob>=0.5:
            topic_id = row['topic'] - 1
            prob = row['topic prob']
            num_signatures = row['area_signature_count']
            topic_weighting[topic_id] += num_signatures * prob

    top_topics.update({
            "Region": region,
            "Year": year,
            "Topic 1 Count": topic_weighting[0],
            "Topic 2 Count": topic_weighting[1],
            "Topic 3 Count": topic_weighting[2],
            "Topic 4 Count": topic_weighting[3],
            "Topic 5 Count": topic_weighting[4],
            "Topic 6 Count": topic_weighting[5],
            "Topic 7 Count": topic_weighting[6],
            "Topic 8 Count": topic_weighting[7],
            "Topic 9 Count": topic_weighting[8],
            "Topic 10 Count": topic_weighting[9],
            "Topic 11 Count": topic_weighting[10],
            "Topic 12 Count": topic_weighting[11],
            "Topic 13 Count": topic_weighting[12],
            "Topic 14 Count": topic_weighting[13],
            "Topic 15 Count": topic_weighting[14],
            "Topic 16 Count": topic_weighting[15],
            "Topic 17 Count": topic_weighting[16],
            "Topic 18 Count": topic_weighting[17],
            })
        
    return top_topics

The above function will be tested on a small subset of 10 petitions first to see if the correct weighting has been given to each topic.

In [40]:
# testing on a small subset first
sample_df.head(10)

,petition_id,url,details,created_at,signature_count,parliament,text,clean_text
21882,565889,https://petition.parliament.uk/archived/petiti...,"On the 1st January 2021, the Government wisely...",2021-01-01 02:14:26.859000+00:00,1079,3,Abolish the 5% VAT applied to domestic gas and...,"[abolish, vat, applied, domestic, gas, electri..."
21883,565890,https://petition.parliament.uk/archived/petiti...,This is the best tv program ever made,2021-01-01 02:22:21.434000+00:00,21,3,"Bring back the Inbetweeners, Bring back the In...","[inbetweeners, inbetweeners, tv, program]"
21885,565898,https://petition.parliament.uk/archived/petiti...,Get out of lockdown and back to a normal econo...,2021-01-01 06:06:40.913000+00:00,21,3,Implement a 24/7 COVID-19 vaccination programm...,"[vaccination, programme, vaccinating, faster, ..."
21893,565914,https://petition.parliament.uk/archived/petiti...,Put the policies of opening schools in tier 4 ...,2021-01-01 10:31:23.226000+00:00,21,3,"Close early years settings in tier 4, Early ye...","[setting, tier, practitioner, nursery, centre,..."
21897,565925,https://petition.parliament.uk/archived/petiti...,Nigel Farage has been the single most successf...,2021-01-01 11:32:53.848000+00:00,21,3,"Give Nigel Farage a Knighthood, Give Nigel Far...","[nigel, farage, knighthood, nigel, farage, kni..."
21906,565953,https://petition.parliament.uk/archived/petiti...,Supertrawlers the length of 14 fishing boats s...,2021-01-01 14:17:50.103000+00:00,21,3,"BAN SUPER TRAWLERS FROM BRITISH WATERS, WE WAN...","[super, trawler, water, super, trawler, plunde..."
21907,565958,https://petition.parliament.uk/archived/petiti...,I call for all interactions between care home ...,2021-01-01 14:53:05.304000+00:00,46,3,Require all staff/resident interactions in car...,"[interaction, recorded, neglect, experiencing,..."
21911,565965,https://petition.parliament.uk/archived/petiti...,"Most political leaders, past and present, will...",2021-01-01 16:03:57.405000+00:00,21,3,"A gong for Nigel, Nigel Farage deserves recogn...","[gong, nigel, nigel, farage, deserves, recogni..."
21914,565970,https://petition.parliament.uk/archived/petiti...,Withdrawing from the single market is an act o...,2021-01-01 16:34:09.670000+00:00,16354,3,Apply to join the European Union Single Market...,"[join, eu, join, eu, withdrawing, sabotage, po..."
21919,565978,https://petition.parliament.uk/archived/petiti...,The Government should extend the date by which...,2021-01-01 17:25:30.756000+00:00,394,3,Extend EU Settlement Scheme for those affected...,"[eu, settlement, eu, settlement, started, dece..."


In [ ]:
# looping through petitions in sample and finding topic assignment and signature count to manually calculate weighting
for document_num in range(len(sample_df.head(10))):
    print(sample_df.iloc[document_num].petition_id, sample_df.iloc[document_num].signature_count, 
          sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1]))

565889 1079 [(11, 0.520403), (13, 0.37798527), (8, 0.057243753)]
565890 21 [(16, 0.89319783)]
565898 21 [(6, 0.48352802), (8, 0.40935135), (0, 0.06500772)]
565914 21 [(6, 0.94393355)]
565925 21 [(5, 0.929505)]
565953 21 [(2, 0.43528447), (11, 0.3353779), (16, 0.12864113), (3, 0.06526226)]
565958 46 [(12, 0.6857834), (7, 0.257104)]
565965 21 [(11, 0.9459123)]
565970 16354 [(16, 0.93952477)]
565978 394 [(9, 0.8041087), (10, 0.05963967), (11, 0.05169891), (16, 0.04413736)]


In [ ]:
# using the find_topic_prob and calculate_weighting_overall functions to see what results are found to compare to above - note that the topics above should be
# 1 higher than written i.e. topic 0 is actually topic 1 - this is a continuation from previuos where changing from starting point of 0 to 1 to make topic ascription more readable
# (was a bit of a mistake as has lead to confusion where switching between the two)
topic_df, errors = find_topic_prob(sample_df.head(10), model, corpus)
print(f"Errors: {errors}")
topics = calculate_weighting_overall(topic_df, 2021)
topics


Processing Summary:
Total documents: 10
Successfully processed: 10
Errors: 0
Errors: []


({'Year': 2021,
  'Topic 1 Count': 0,
  'Topic 2 Count': 0,
  'Topic 3 Count': 0,
  'Topic 4 Count': 0,
  'Topic 5 Count': 0,
  'Topic 6 Count': 19.519603550434113,
  'Topic 7 Count': 19.822604477405548,
  'Topic 8 Count': 0,
  'Topic 9 Count': 0,
  'Topic 10 Count': 316.6130977869034,
  'Topic 11 Count': 0,
  'Topic 12 Count': 580.9312750697136,
  'Topic 13 Count': 31.557419776916504,
  'Topic 14 Count': 0,
  'Topic 15 Count': 0,
  'Topic 16 Count': 0,
  'Topic 17 Count': 15383.745239555836,
  'Topic 18 Count': 0},
 [0,
  0,
  0,
  0,
  0,
  19.519603550434113,
  19.822604477405548,
  0,
  0,
  316.6130977869034,
  0,
  580.9312750697136,
  31.557419776916504,
  0,
  0,
  0,
  15383.745239555836,
  0])

Test provides accurate results.


In [ ]:
def generate_quality_report(year):
    """
    Comprehensive quality report for a given year's model using some of the previous tests
    :param year: year as a whole number between 2015-2025 to test that years data
    """
    dictionary, corpus, model = load_model(year)
    file = load_file(year)
    
    report = {
        'year': year,
        'total_documents': len(file),
        'corpus_size': len(corpus),
        'alignment_check': len(corpus) == len(file),
    }
    
    # 1. Assignment clarity
    unclear_count = 0
    high_confidence_count = 0
    
    for doc_num in range(min(len(file), len(corpus))):
        val = validate_topic_assignment(model, corpus, doc_num)
        if not val['is_clear_assignment']:
            unclear_count += 1
        if val['dominant_prob'] > 0.7:
            high_confidence_count += 1
    
    report['unclear_assignments_pct'] = (unclear_count / len(corpus)) * 100
    report['high_confidence_pct'] = (high_confidence_count / len(corpus)) * 100
    
    # 2. Topic distribution balance
    topic_df, errors = find_topic_prob(file, model, corpus)
    topic_weighting = calculate_weighting_overall(topic_df, year)[1]
    report['topic_distribution_std'] = np.std(topic_weighting)
    report['topic_distribution_cv'] = np.std(topic_weighting) / np.mean(topic_weighting)
       
    # 3. Coverage
    topic_df, errors = find_topic_prob(file, model, corpus)
    topic_weighting = calculate_weighting_overall(topic_df, year)[1]
    top_3_coverage = sum(sorted(topic_weighting, reverse=True)[:3]) / sum(topic_weighting)
    report['top_3_topics_coverage_pct'] = top_3_coverage * 100
    
    return report

# Generate reports for all years
quality_reports = []
for year in possible_years:
    try:
        report = generate_quality_report(year)
        quality_reports.append(report)
        print(f"\n{year} Quality Report:")
        for key, value in report.items():
            print(f"  {key}: {value}")
    except Exception as e:
        print(f"Error processing {year}: {e}")

quality_df = pd.DataFrame(quality_reports)
quality_df.to_csv("data/model_quality_reports.csv", index=False)


Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

2015 Quality Report:
  year: 2015
  total_documents: 2037
  corpus_size: 2038
  alignment_check: False
  unclear_assignments_pct: 25.122669283611383
  high_confidence_pct: 51.37389597644749
  topic_distribution_std: 370392.5102370968
  topic_distribution_cv: 1.0302549023989553
  top_3_topics_coverage_pct: 48.06588097112851

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

2016 Quality Report:
  year: 2016
  total_documents: 5075
  corpus_size: 5075
  alignment_check: True
  unclear_assignments_pct: 26.482758620689655
  high_confidence_pct: 38.64039408866995
  topic_distribution_std: 1065920.664228755
  topic_distribution_cv: 1.5888200884029853
  top_3_topics_coverage_pct: 66.02082881242787

Processing Summary:



## Result extraction

In [44]:
# creating empty dataframe for the signature count calculations
top_topics_df = pd.DataFrame(columns=["Year", "Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", "Topic 5 Count", 
                                      "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", "Topic 9 Count", "Topic 10 Count", 
                                      "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", "Topic 15 Count", 
                                      "Topic 16 Count","Topic 17 Count","Topic 18 Count"])


In [ ]:
possible_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# looping through years to get results
for year in possible_years:
    # loading files and corpus for year
    dictionary, corpus, model = load_model(year)
    file = load_file(year)

    # split petitions into topics and find topic probabilities
    topic_df, errors = find_topic_prob(file, model, corpus)
    # calculate topic weightings and add to empty dataframe
    top_topics_df.loc[len(top_topics_df)] = calculate_weighting_overall(topic_df, year)[0]

top_topics_df.head()
    


Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

Processing Summary:
Total documents: 3330
Successfully processed: 3330
Errors: 0

Processing Summary:
Total documents: 4287
Successfully processed: 4287
Errors: 0

Processing Summary:
Total documents: 5173
Successfully processed: 5173
Errors: 0

Processing Summary:
Total documents: 8381
Successfully processed: 8381
Errors: 0

Processing Summary:
Total documents: 5387
Successfully processed: 5387
Errors: 0

Processing Summary:
Total documents: 3030
Successfully processed: 3030
Errors: 0

Processing Summary:
Total documents: 3061
Successfully processed: 3061
Errors: 0

Processing Summary:
Total documents: 2339
Successfully processed: 2339
Errors: 0

Processing Summary:
Total documents: 1373
Successfully processed: 1373
Errors: 0


,Year,Topic 1 Count,Topic 2 Count,Topic 3 Count,Topic 4 Count,Topic 5 Count,Topic 6 Count,Topic 7 Count,Topic 8 Count,Topic 9 Count,Topic 10 Count,Topic 11 Count,Topic 12 Count,Topic 13 Count,Topic 14 Count,Topic 15 Count,Topic 16 Count,Topic 17 Count,Topic 18 Count
0,2015,3.274649e+05,1.506466e+05,3.583197e+05,390506.971294,782081.316779,732486.226183,3.452238e+05,438089.997794,1.271677e+06,302557.228661,1.057370e+06,314800.296240,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.0
1,2016,1.999118e+05,3.279705e+05,1.030979e+06,321886.800299,515178.394616,427551.136127,2.390240e+06,89464.788257,4.409735e+05,497559.206293,4.843803e+05,237181.215241,217804.787352,4.483514e+06,350418.041732,0.000000,0.000000,0.0
2,2017,3.915759e+05,5.012791e+05,6.442343e+04,462947.890285,468082.437188,366220.917824,2.183366e+05,196885.782946,1.706217e+05,398412.638119,3.946191e+05,186936.067753,98978.067763,4.084061e+05,0.000000,0.000000,0.000000,0.0
3,2018,2.174677e+05,2.099409e+06,4.663072e+04,53131.861702,472420.330008,300816.061431,5.084045e+05,492861.076197,2.189498e+05,426024.020807,5.908039e+05,169774.227505,265518.767020,1.745328e+05,207418.540867,414559.486657,0.000000,0.0
4,2019,4.128688e+06,8.328687e+04,9.443090e+04,23749.919754,493335.350890,212568.785333,5.808241e+05,671804.695896,9.252818e+04,206294.136862,1.870800e+05,89362.300097,135605.678503,2.855256e+05,502141.755027,60339.164844,228277.701998,0.0


In [47]:
top_topics_df.to_csv("data/Top_topics.csv")

Next will be breaking up petitions into region to understand the geographical story:

In [ ]:
def merge_petitions_region(file):
    """ 
    returns a file with petition signatures for a given region
    :param file: the filtered petition year file
    :output: a file containing region information along with signature count for each petition
    """
    # merge petitions with region and create region list
    petitions_by_region = file.set_index("petition_id").join(
    regions_df.set_index("petition_id"), 
    lsuffix='_x', 
    rsuffix='_y', 
    how='left').reset_index()
    # renaming columns to make clearer
    petitions_by_region = petitions_by_region.rename(columns={"signature_count_y": "area_signature_count", "signature_count_x": "signature_count"})
    
    return petitions_by_region

In [ ]:
# creating list of region names to loop through
regions = regions_df['name'].unique()

# empty dataframe for signature count by topic by region
top_topics_region_df = pd.DataFrame(columns=["Region", "Year", "Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", 
                                             "Topic 5 Count", "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", "Topic 9 Count", 
                                             "Topic 10 Count", "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", 
                                             "Topic 15 Count", "Topic 16 Count", "Topic 17 Count","Topic 18 Count"])
   
# looping through all years to extract results
for year in possible_years:

    # loading model and file
    dictionary, corpus, model = load_model(year)
    file = load_file(year)    

    # finding topic probabilities and then merging with region file
    topic_df, errors = find_topic_prob(file, model, corpus)
    region_file = merge_petitions_region(topic_df)
        
    # use region list as an index and for region in regions, filter for that region and apply the above, give the top 3 topics
    for region in regions:
        top_topics_region_df.loc[len(top_topics_region_df)] = calculate_weighting_region(region_file, year, region)


Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

Processing Summary:
Total documents: 3330
Successfully processed: 3330
Errors: 0

Processing Summary:
Total documents: 4287
Successfully processed: 4287
Errors: 0

Processing Summary:
Total documents: 5173
Successfully processed: 5173
Errors: 0

Processing Summary:
Total documents: 8381
Successfully processed: 8381
Errors: 0

Processing Summary:
Total documents: 5387
Successfully processed: 5387
Errors: 0

Processing Summary:
Total documents: 3030
Successfully processed: 3030
Errors: 0

Processing Summary:
Total documents: 3061
Successfully processed: 3061
Errors: 0

Processing Summary:
Total documents: 2339
Successfully processed: 2339
Errors: 0

Processing Summary:
Total documents: 1373
Successfully processed: 1373
Errors: 0


In [50]:
top_topics_region_df.to_csv("data/Top_topics_by_region.csv")

In [ ]:
def merge_consituency_petitions(file):
    """ 
    returns a file with petition signatures for a given constituency
    :param file: the filtered petition year file
    :output: a file containing constituency information along with signature count for each petition
    """
    # merge petitions with constituency and creating constituency list
    petitions_by_constituency = file.set_index("petition_id").join(
    constituencies_df.set_index("petition_id"), 
    lsuffix='_x', 
    rsuffix='_y', 
    how='left').reset_index()

    # renaming columns to make clearer
    petitions_by_constituency = petitions_by_constituency.rename(columns={
        "signature_count_y": "area_signature_count", 
        "signature_count_x": "signature_count"})

    return petitions_by_constituency


In [ ]:

# list of constituency names to loop through
constituencies = constituencies_df['name'].unique()

# empty dataframe to calculate signature count by topic by constituency and year
top_topics_con_df = pd.DataFrame(columns=["Constituency", "Year", "Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", 
                                             "Topic 5 Count", "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", "Topic 9 Count", 
                                             "Topic 10 Count", "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", 
                                             "Topic 15 Count", "Topic 16 Count", "Topic 17 Count","Topic 18 Count"])

# to keep track o how many consituencies have been looked up (more than 650 as constituency boundaries have changed over time)
con_count = 0

# looping through years to extract all results
for year in possible_years:
    # load model and file and then merge
    dictionary, corpus, model = load_model(year)
    file = load_file(year)

    # finding topic probabilities and then merging with constituency file
    topic_df, errors = find_topic_prob(file, model, corpus)
    con_file = merge_consituency_petitions(topic_df)
        
    # use constituency list as an index and for constituency in constituencies, filter for that constituency and calculate weighting
    for con in constituencies:
        top_topics_con_df.loc[len(top_topics_con_df)] = calculate_weighting_con(con_file, year, con)
        
        # keeping track of how many constituencies have been processed
        con_count += 1
        print('Constituency count:', con_count)



Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0
Constituency count: 1
Constituency count: 2
Constituency count: 3
Constituency count: 4
Constituency count: 5
Constituency count: 6
Constituency count: 7
Constituency count: 8
Constituency count: 9
Constituency count: 10
Constituency count: 11
Constituency count: 12
Constituency count: 13
Constituency count: 14
Constituency count: 15
Constituency count: 16
Constituency count: 17
Constituency count: 18
Constituency count: 19
Constituency count: 20
Constituency count: 21
Constituency count: 22
Constituency count: 23
Constituency count: 24
Constituency count: 25
Constituency count: 26
Constituency count: 27
Constituency count: 28
Constituency count: 29
Constituency count: 30
Constituency count: 31
Constituency count: 32
Constituency count: 33
Constituency count: 34
Constituency count: 35
Constituency count: 36
Constituency count: 37
Constituency count: 38
Constituency count: 39
Constituency count: 40
Constit

In [53]:
top_topics_con_df.to_csv("data/Top_topics_by_constituency.csv")

In [ ]:
# calling previous split topic function and merging with constituency file to find top petitions for a given topic for a given constituency and understand the data better
dictionary, corpus, model = load_model(2023)
petitions = load_file(2023)
extracted_df = split_topic_petitions(petitions, 2, model, corpus)
merge_df = merge_consituency_petitions(extracted_df)
merge_df.to_csv("data/2023_02.csv")


Score: 0.5686280131340027	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.918750524520874	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.9354913234710693	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.9190236330032349	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.9181538820266724	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.583587110042572	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.7494125366210938	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.5679603219032288	 
Topic: 0.034*"domestic" + 0.034*"sexual" + 0.032*"sex" + 0.032*"asylum" + 0.031*"offender"

Score: 0.8127616047859192	 
Topic: 0.034*

For reference for some reading on similar topics which have assisted in setting aims for the project and understanding the model, see below.

### Bibliography
- Vidgen B, Yasseri T. What, when and where of petitions submitted to the UK government during a time of chaos. Policy sciences. 2020;53(3):535-557. doi:10.1007/s11077-020-09395-y
- Bicalho P, Pita M, Pedrosa G, Lacerda A, Pappa GL. A general framework to expand short text for topic modeling. Information sciences. 2017;393:66-81. doi:10.1016/j.ins.2017.02.007
- Hagen L. Content analysis of e-petitions with topic modeling: How to train and evaluate LDA models? Information processing & management. 2018;54(6):1292-1307. doi:10.1016/j.ipm.2018.05.006